# Démo — Segmentation par couleur d'une carte topographique

**PFA — Mohamed GHARBI**

Pipeline : extraction des couches colorimétriques (eau, végétation, courbes, routes rouges) **sans deep learning**, OpenCV uniquement.

Ce notebook fonctionne **dans Google Colab** (GPU T4 gratuit, 15 Go VRAM, pas de soucis DLL Windows) **ou en local sous Anaconda**. La première cellule détecte automatiquement où tu tournes.

## Setup Colab (sans effet en local)

Sur Colab : pip-install les dépendances + clone le projet depuis GitHub.

**Avant de lancer**, ouvre le menu `Exécution > Modifier le type d'exécution` et choisis **GPU** (T4).

In [ ]:
# === Setup Colab + local ===
# Adapte REPO_URL avec ton repo GitHub si tu utilises Colab.
# Si tu preferes Google Drive, mets USE_DRIVE = True et place le projet
# dans /content/drive/MyDrive/pfa.
REPO_URL  = 'https://github.com/Mohamed-GHARBI/pfa.git'    # <- adapte
BRANCH    = 'main'
USE_DRIVE = False

import sys, subprocess, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('Environnement :', 'Google Colab' if IN_COLAB else 'Local')

if IN_COLAB:
    print('\nInstallation des dependances...')
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install',
        'opencv-python==4.10.0.84', 'scikit-image>=0.22',
        'rasterio>=1.3', 'shapely>=2.0', 'geopandas>=0.14',
        'pyogrio', 'fiona', 'ipywidgets>=8.1'])
    print('OK')

    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_ROOT = Path('/content/drive/MyDrive/pfa')
    else:
        PROJECT_ROOT = Path('/content/pfa')
        if not (PROJECT_ROOT / 'pipeline').exists():
            print(f'\nClonage du repo : {REPO_URL}')
            subprocess.check_call(['git', 'clone', '--depth', '1',
                                     '--branch', BRANCH, REPO_URL,
                                     str(PROJECT_ROOT)])
            print('OK')
else:
    PROJECT_ROOT = Path('..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT / 'notebooks' if (PROJECT_ROOT / 'notebooks').exists() else PROJECT_ROOT)
print(f'\nPROJECT_ROOT : {PROJECT_ROOT}')
print(f'cwd          : {os.getcwd()}')

## Vérification de l'environnement

In [ ]:
import platform
print(f'Python : {platform.python_version()}')

_required = {
    'numpy': 'numpy', 'cv2': 'opencv', 'matplotlib': 'matplotlib',
    'skimage': 'scikit-image', 'shapely': 'shapely', 'geopandas': 'geopandas',
    'rasterio': 'rasterio',
}
missing = []
for mod, pkg in _required.items():
    try:
        m = __import__(mod); v = getattr(m, '__version__', '?')
        print(f'  OK   {mod:12s} {v}')
    except ImportError:
        print(f'  FAIL {mod:12s} ({pkg})'); missing.append(pkg)

# GPU
print()
try:
    import torch
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU OK : {p.name}  ({p.total_memory/1e9:.1f} Go VRAM, '
              f'CUDA {torch.version.cuda})')
    else:
        print('  CPU uniquement (pas de GPU CUDA detectee)')
        if IN_COLAB:
            print('  -> Active la GPU : Execution > Modifier type d\'execution > T4 GPU')
except ImportError:
    print('  PyTorch non installe (pas critique pour ce notebook)')

## Données — où est ta carte ?

**Local Anaconda** : place ta carte dans `data/raw/carte_test.png`.

**Colab** : 3 options :
1. Si tu as cloné depuis GitHub, la carte est déjà dans `data/raw/`.
2. Sinon, **upload manuel** : clique sur l'icône dossier 📁 à gauche, puis glisse-dépose ton fichier dans `pfa/data/raw/`.
3. Ou utilise Drive : mets `USE_DRIVE=True` plus haut + carte dans `MyDrive/pfa/data/raw/`.

In [ ]:
# Imports pipeline
import numpy as np
import matplotlib.pyplot as plt

from pipeline import preprocessing as prep
from pipeline import color_segmentation as colseg
from pipeline import vectorization as vec

INPUT_PATH = str(PROJECT_ROOT / 'data' / 'raw' / 'carte_test.png')
print('Carte attendue :', INPUT_PATH)
print('Existe :', os.path.exists(INPUT_PATH))

## 1. Chargement, prétraitement et recadrage

`max_dimension=2400` : redimensionne en amont pour éviter les OOM. Pour le 1:50000 c'est largement suffisant.

In [ ]:
image_bgr, image_hsv, used_bbox = prep.preprocess_with_crop(
    INPUT_PATH,
    auto_crop=True,
    max_dimension=2400,
    # manual_bbox=(180, 220, 2400, 1800),  # decommente si auto-crop rate
)
image_rgb = prep.to_rgb(image_bgr)

print('Cadre cartographique utilise :', used_bbox)
print('Dimensions carte (recadree)  :', image_bgr.shape)

plt.figure(figsize=(12, 8))
plt.imshow(image_rgb)
plt.title(f'Carte recadree {image_bgr.shape[1]}x{image_bgr.shape[0]} px')
plt.axis('off'); plt.show()

## 2. Extraction des couches couleur

In [ ]:
import gc; gc.collect()

layers = colseg.extract_all_color_layers(image_hsv)
print('Couches extraites :')
for name, mask in layers.items():
    pct = colseg.coverage_percent(mask)
    print(f'  {name:12s} -> {pct:5.2f}% des pixels')

## 3. Visualisation des couches

In [ ]:
n = len(layers)
fig, axes = plt.subplots(1, n, figsize=(5*n, 6))
if n == 1: axes = [axes]
for ax, (name, mask) in zip(axes, layers.items()):
    ax.imshow(mask, cmap='gray'); ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Superposition sur la carte originale

In [ ]:
COLORS = {'water':(0,120,255), 'vegetation':(0,180,0),
          'contours':(160,90,30), 'red_roads':(255,0,0)}
overlay = image_rgb.copy()
for name, mask in layers.items():
    overlay[mask > 0] = COLORS.get(name, (255,255,0))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9))
ax1.imshow(image_rgb); ax1.set_title('Original');         ax1.axis('off')
ax2.imshow(overlay);   ax2.set_title('Couches extraites'); ax2.axis('off')
plt.show()

## 5. Vectorisation et export en GeoJSON

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name in ['water', 'vegetation', 'red_roads']:
    mask = layers.get(name)
    if mask is None or not mask.any(): continue
    polys = vec.mask_to_polygons(mask, min_area_px=30)
    if polys:
        gdf = vec.to_geodataframe(polys, layer_name=name)
        out = OUTPUT_DIR / f'{name}.geojson'
        vec.save_geojson(gdf, out)
        print(f'  -> {out}  ({len(polys)} polygones)')

contour_mask = layers.get('contours')
if contour_mask is not None and contour_mask.any():
    lines = vec.skeleton_to_lines(contour_mask)
    if lines:
        gdf = vec.to_geodataframe(lines, layer_name='contours')
        out = OUTPUT_DIR / 'contours.geojson'
        vec.save_geojson(gdf, out)
        print(f'  -> {out}  ({len(lines)} lignes)')

print('\nFichiers dans', OUTPUT_DIR)
if IN_COLAB:
    print("Pour les telecharger : icone dossier a gauche -> menu '...' -> Telecharger")
    print("Ou pour tout zipper : !zip -r /content/processed.zip", str(OUTPUT_DIR))

## En cas de plantage

**Kernel meurt** -> reduis `max_dimension=2000` ou `1600`, ou `denoise_on=False` dans `preprocess_with_crop`.

**ModuleNotFoundError pipeline** apres restart -> relance la cellule de Setup en premier.

**Sur Colab** : si tu modifies un fichier `pipeline/*.py`, fais `Execution > Redemarrer l'execution` (sinon les changements ne sont pas pris en compte).

## Prochaines etapes

1. Calibration HSV - `02_hsv_calibration.ipynb`
2. Georeferencement automatique - `03_georeferencing.ipynb`
3. Segmentation U-Net - `pipeline/semantic_segmentation.py` (GPU auto)